# Cluster-Level Sentiment Aggregation
Merge the full-history SEC sentiment scores (Fatima's deliverable) with the full-history cluster assignments (Grace's deliverable) to build the cluster-level sentiment table for downstream anomaly detection.

In [14]:
import pandas as pd
from pathlib import Path

# 1. Setup paths
root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent
features_dir = root / "data" / "processed" / "features"

## 1. Load the Datasets
Load the outputs from the Week 3 NLP and clustering pipelines. We ensure the date columns are cast to `datetime` objects to prevent merging issues.

In [ ]:
# Load daily sentiment features
sentiment_path = features_dir / "sec_sentiment_daily.parquet" 
sentiment_df = pd.read_parquet(sentiment_path)

# Load community memberships from week 3 output directory
grace_output_dir = root / "notebooks" / "outputs" / "grace_week3"
cluster_path = grace_output_dir / "community_memberships.csv"
cluster_df = pd.read_csv(cluster_path)

# Standardize dates for merging
sentiment_df['date'] = pd.to_datetime(sentiment_df['date'])
cluster_df['date'] = pd.to_datetime(cluster_df['date'])

print(f"Loaded {len(sentiment_df)} sentiment records and {len(cluster_df)} cluster records.")

Loaded 2510 sentiment records and 95601 cluster records.


## 2. Merge Sentiment and Cluster Data
We perform an inner join on `date` and `ticker` so that every stock has both an assigned cluster and a daily sentiment score.

In [16]:
merged_df = pd.merge(
    sentiment_df, 
    cluster_df, 
    on=['date', 'ticker'], 
    how='inner'
)

display(merged_df.head())

,date,ticker,section_group,average_sentiment,tone_change,paragraphs,source_filing_trading_date,community,sector
0,2022-01-31,AAPL,MD&A,0.108696,NaN,46.0,2022-01-28,2,NaN
1,2022-03-01,AAPL,MD&A,0.108696,NaN,46.0,2022-01-28,1,NaN
2,2022-03-29,AAPL,MD&A,0.108696,NaN,46.0,2022-01-28,1,NaN
3,2022-04-27,AAPL,MD&A,0.108696,NaN,46.0,2022-01-28,1,NaN
4,2022-05-25,AAPL,MD&A,0.120000,0.011304,50.0,2022-04-29,1,NaN


## 3. Aggregate Sentiment by Cluster
Group the merged dataset by `date` and `cluster_id`. We calculate the mean sentiment score to gauge the overall mood of the cluster, as well as the paragraph count to measure conviction/volume.

In [ ]:
# 1. Rename column to match the expected downstream format
merged_df = merged_df.rename(columns={'community': 'cluster_id'})

# 2. Perform the aggregation
cluster_sentiment = merged_df.groupby(['date', 'cluster_id']).agg(
    mean_sentiment=('average_sentiment', 'mean'),
    paragraph_count=('paragraphs', 'sum') 
).reset_index()

display(cluster_sentiment.head())

,date,cluster_id,mean_sentiment,paragraph_count
0,2022-01-31,2,-0.137820,201.0
1,2022-03-01,1,-0.187034,660.0
2,2022-03-29,1,-0.216445,867.0
3,2022-04-27,1,-0.239458,807.0
4,2022-05-25,1,-0.347471,536.0


## 4. Save Final Feature Table
Save the aggregated table to the features directory so Rishi can merge it into the frozen feature table.

In [21]:
output_path = features_dir / "cluster_level_sentiment.parquet"
cluster_sentiment.to_parquet(output_path, index=False)

print(f"Successfully saved cluster sentiment table to: {output_path}")

Successfully saved cluster sentiment table to: /Users/broy/Documents/GitHub/RGBFS/data/processed/features/cluster_level_sentiment.parquet
